# PE6201 A2 D6 - Project A Cost-to-Serve Model

Run this notebook from the repository root or from `cost_model/`.

It reads the committed final evaluation summaries, the verified price table, the D2(c) measurement,
and the configured caps from the repository. It then calculates the D6 baseline, sensitivity,
break-even, cost ledger, caps, and a report-ready summary.

Manual inputs are limited to deployment assumptions that are not measured in the repository:

- `FIXED_MONTHLY_USD`: layer 3 fixed monthly cost. Default is `0.0` because no measured layer 3 cost is committed.
- `MONTHLY_LIMIT_USD_PER_USER`: monthly per-user limit. Default is `None` because no such cap is configured.

## Section 1 - Load Repository Inputs

Inputs used here:

- `src/backends.py`: verified `PRICES` table.
- `results/evaluations/*final.summary.json`: measured model pass rates and token counts.
- `evaluation/d2c_run.json`: sequential vs parallel measurement for Lever 2.
- `src/loop.py`: configured caps.
- `cost_model/D6_input_audit.md`: input audit and source notes.

In [ ]:
import ast
import json
from pathlib import Path


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "backends.py").exists() and (candidate / "results" / "evaluations").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or cost_model/.")


REPO = find_repo_root()
RESULTS_DIR = REPO / "results" / "evaluations"
OUTPUT_JSON = REPO / "results" / "d6_cost_model.json"

PROJECT_A_VOLUME = 8_000
FAILURE_USD = 7.60
RETRIEVAL_USD = 0.0
TOOL_USD = 0.0

# Layer 3 deployment assumption. Set by the team as an experience-based estimate.
FIXED_MONTHLY_USD = 500.0
FIXED_MONTHLY_NOTE = "experience-based deployment assumption"

# Monthly per-user deployment cap. Set by the team as an experience-based estimate.
MONTHLY_LIMIT_USD_PER_USER = 1_000.0
MONTHLY_LIMIT_NOTE = "experience-based deployment assumption"

print(f"Repository: {REPO}")
print(f"Project A volume: {PROJECT_A_VOLUME:,} claims/month")
print(f"Failure cost: US${FAILURE_USD:.2f} per failed claim")
print(f"Layer 3 fixed monthly cost: US${FIXED_MONTHLY_USD:.2f} ({FIXED_MONTHLY_NOTE})")
print(f"Monthly per-user limit: US${MONTHLY_LIMIT_USD_PER_USER:.2f} ({MONTHLY_LIMIT_NOTE})")

## Section 2 - Prices and Final Runs

Use final combined pass rate for D6. Do not use code-only or decision-only rates for the cost model.

The `openai/gpt-4o-mini` v1 row is retained as prompt-comparison evidence, but excluded from the v2 live battery model comparison.

In [ ]:
def load_prices():
    src = (REPO / "src" / "backends.py").read_text(encoding="utf-8")
    mod = ast.parse(src)
    for node in mod.body:
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == "PRICES":
            return ast.literal_eval(node.value)
    raise RuntimeError("PRICES not found in src/backends.py")


PRICES_RAW = load_prices()
PRICES = {model: {"in": price[0], "out": price[1]} for model, price in PRICES_RAW.items()}


def load_summary(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    model = data["model"]
    if model not in PRICES:
        raise RuntimeError(f"No price for {model} from {path.name}")
    trials = data["code_checked_trial_count"]
    row = {
        "file": path.name,
        "model": model,
        "run_id": data["run_id"],
        "backend": data["backend"],
        "pass_count": data["final_passed_trial_count"],
        "trials": trials,
        "success_rate": data["final_combined_pass_rate"],
        "input_tokens_total": data["total_input_tokens"],
        "output_tokens_total": data["total_output_tokens"],
        "input_tokens_per_run": data["total_input_tokens"] / trials,
        "output_tokens_per_run": data["total_output_tokens"] / trials,
        "median_turns": data["median_turns"],
        "caps_fired": data["cap_fired_count"],
        "combined_score_status": data["combined_score_status"],
        "missing_judgements": data["missing_judgement_trial_count"],
        "price_in": PRICES[model]["in"],
        "price_out": PRICES[model]["out"],
    }
    row["use"] = "scripted_baseline" if row["backend"] == "scripted" else "live_v2_battery"
    if "-v1-" in path.name:
        row["use"] = "v1_prompt_comparison_only"
    return row


runs = [load_summary(p) for p in sorted(RESULTS_DIR.glob("*final.summary.json"))]
for row in runs:
    if row["combined_score_status"] != "complete" or row["missing_judgements"] != 0:
        raise RuntimeError(f"Incomplete score in {row['file']}")

print(f"Loaded {len(runs)} final summaries")
print(f"{'use':<28}{'model':<42}{'pass':>10}{'input/run':>12}{'output/run':>12}{'caps':>8}")
for r in runs:
    print(f"{r['use']:<28}{r['model']:<42}{r['pass_count']:>3}/{r['trials']:<6}{r['input_tokens_per_run']:>12,.0f}{r['output_tokens_per_run']:>12,.0f}{r['caps_fired']:>8}")

## Section 3 - Three-Layer Cost Model

Baseline formula:

`layer 1 variable = input*price_in + output*price_out + retrieval/tool fees`

`layer 2 fallback = (1 - success_rate) * failure_cost`

`monthly = cost per successful task * volume + layer 3`

In [ ]:
def variable_cost(row):
    return (
        row["input_tokens_per_run"] / 1_000_000 * row["price_in"]
        + row["output_tokens_per_run"] / 1_000_000 * row["price_out"]
        + RETRIEVAL_USD
        + TOOL_USD
    )


def cost_per_successful_task(layer_1, success_rate):
    return layer_1 + (1 - success_rate) * FAILURE_USD


def money(x, places=4):
    return f"US${x:,.{places}f}"


def pct(x):
    return f"{x:.2%}"


model_rows = []
for r in runs:
    layer_1 = variable_cost(r)
    layer_2 = (1 - r["success_rate"]) * FAILURE_USD
    per_success = layer_1 + layer_2
    monthly = per_success * PROJECT_A_VOLUME + FIXED_MONTHLY_USD
    out = dict(r)
    out.update({
        "layer_1_variable_usd": layer_1,
        "layer_2_fallback_usd": layer_2,
        "layer_3_fixed_monthly_usd": FIXED_MONTHLY_USD,
        "cost_per_successful_claim_usd": per_success,
        "monthly_total_usd": monthly,
    })
    model_rows.append(out)

print(f"{'use':<28}{'model':<42}{'success':>10}{'layer1':>12}{'layer2':>12}{'per success':>14}{'monthly':>14}")
for r in sorted(model_rows, key=lambda x: (x["use"], x["monthly_total_usd"])):
    print(f"{r['use']:<28}{r['model']:<42}{pct(r['success_rate']):>10}{money(r['layer_1_variable_usd']):>12}{money(r['layer_2_fallback_usd']):>12}{money(r['cost_per_successful_claim_usd']):>14}{money(r['monthly_total_usd'], 2):>14}")

## Section 4 - Headline Comparison and Break-Even

For the D6 model-choice comparison, use v2 live battery rows. The notebook picks:

- `cheap`: best all-in cheap-tier v2 live row.
- `mid`: `mistralai/mistral-medium-3`, the mid-tier v2 live row.

This keeps the comparison on the same v2 prompt while preserving the two-tier question.

In [ ]:
live_v2 = [r for r in model_rows if r["use"] == "live_v2_battery"]
cheap_model_ids = {
    "openai/gpt-4o-mini",
    "google/gemini-2.5-flash-lite",
    "meta-llama/llama-3.3-70b-instruct",
    "deepseek/deepseek-chat",
}
cheap_rows = [r for r in live_v2 if r["model"] in cheap_model_ids]
mid_rows = [r for r in live_v2 if r["model"] == "mistralai/mistral-medium-3"]

headline_cheap = min(cheap_rows, key=lambda r: r["cost_per_successful_claim_usd"])
headline_mid = mid_rows[0]

C = headline_cheap["layer_1_variable_usd"]
E = headline_mid["cost_per_successful_claim_usd"]
F = FAILURE_USD
break_even = max(0.0, min(1.0, 1 - (E - C) / F))
gap = headline_cheap["success_rate"] - break_even
monthly_saving = headline_mid["monthly_total_usd"] - headline_cheap["monthly_total_usd"]

headline = {
    "cheap_model": headline_cheap["model"],
    "mid_model": headline_mid["model"],
    "cheap_success_rate": headline_cheap["success_rate"],
    "mid_success_rate": headline_mid["success_rate"],
    "break_even_success_rate_for_cheap": break_even,
    "cheap_break_even_gap": gap,
    "monthly_saving_vs_mid_usd": monthly_saving,
    "annual_saving_vs_mid_usd": monthly_saving * 12,
}

print("Headline live v2 comparison")
print(f"Cheap row: {headline_cheap['model']} at {pct(headline_cheap['success_rate'])}, {money(headline_cheap['cost_per_successful_claim_usd'])}/successful claim")
print(f"Mid row:   {headline_mid['model']} at {pct(headline_mid['success_rate'])}, {money(headline_mid['cost_per_successful_claim_usd'])}/successful claim")
print(f"Break-even success rate for cheap: {pct(break_even)}")
if gap >= 0:
    print(f"Conclusion: cheap clears break-even by {gap:.2%}.")
else:
    print(f"Conclusion: cheap falls short of break-even by {-gap:.2%}.")
print(f"Monthly difference cheap vs mid: {money(monthly_saving, 2)}")
print(f"Annual difference cheap vs mid: {money(monthly_saving * 12, 2)}")

## Section 5 - Sensitivity

D6 requires cost per successful task across success rate +/- 10 percentage points. The robustness check below compares the headline cheap row against the mid-tier row across that range.

In [ ]:
def sensitivity_rows(row, spread=0.10, step=0.05):
    rates = []
    r = max(0.0, row["success_rate"] - spread)
    end = min(1.0, row["success_rate"] + spread)
    while r <= end + 1e-9:
        layer_1 = row["layer_1_variable_usd"]
        per = cost_per_successful_task(layer_1, r)
        rates.append({"success_rate": r, "cost_per_successful_claim_usd": per, "monthly_total_usd": per * PROJECT_A_VOLUME + FIXED_MONTHLY_USD})
        r += step
    return rates

sensitivity = {
    headline_cheap["model"]: sensitivity_rows(headline_cheap),
    headline_mid["model"]: sensitivity_rows(headline_mid),
}

for model, rows in sensitivity.items():
    measured = next(r for r in [headline_cheap, headline_mid] if r["model"] == model)["success_rate"]
    print()
    print(model)
    for row in rows:
        mark = " <- measured" if abs(row["success_rate"] - measured) < 1e-9 else ""
        print(f"  {pct(row['success_rate']):>7}  {money(row['cost_per_successful_claim_usd']):>12}  monthly {money(row['monthly_total_usd'], 2):>12}{mark}")

cheap_worst = max(row["cost_per_successful_claim_usd"] for row in sensitivity[headline_cheap["model"]])
mid_best = min(row["cost_per_successful_claim_usd"] for row in sensitivity[headline_mid["model"]])
robust = cheap_worst < mid_best
print()
print("Robustness conclusion:", "cheap wins across the full +/-10pp range" if robust else "the ranges overlap; conclusion is not robust across the full +/-10pp range")


## Section 6 - Cost Ledger

The four levers are reported separately. Lever 4 is expected to dominate because `F = US$7.60` makes success-rate movement much larger than token-price movement.

In [ ]:
d2c = json.loads((REPO / "evaluation" / "d2c_run.json").read_text(encoding="utf-8"))
lever1 = json.loads((REPO / "evaluation" / "d6_lever1_run.json").read_text(encoding="utf-8"))

lever_2_sequential = d2c["sequential"]
lever_2_by_rule = d2c["by_rule"]
lever_3_descriptor_delta_tokens = 131_216
lever_3_return_shape_delta_tokens = -1_751

one_pp_monthly = 0.01 * FAILURE_USD * PROJECT_A_VOLUME

cost_ledger = {
    "lever_1_tool_block_B": {
        "status": "measured counterfactual prompt-prefix replay; live model behaviour not re-measured",
        "source": "evaluation/d6_lever1_run.json",
        "before": "counterfactual fat tool block with check_required_documents and lookup_member descriptors",
        "after": "shipped tool block",
        "manual_tokens_before": lever1["fat_tool_block"]["manual_estimated_tokens"],
        "manual_tokens_after": lever1["shipped"]["manual_estimated_tokens"],
        "input_tokens_before": lever1["fat_tool_block"]["tokens_in"],
        "input_tokens_after": lever1["shipped"]["tokens_in"],
        "cost_before_usd": lever1["fat_tool_block"]["cost_usd"],
        "cost_after_usd": lever1["shipped"]["cost_usd"],
        "delta_input_tokens_removed": lever1["delta_fat_minus_shipped"]["tokens_in"],
        "delta_cost_usd_removed": lever1["delta_fat_minus_shipped"]["cost_usd"],
        "decisions_before": f"{lever1['fat_tool_block']['decisions_correct']}/{lever1['fat_tool_block']['trials']}",
        "decisions_after": f"{lever1['shipped']['decisions_correct']}/{lever1['shipped']['trials']}",
    },
    "lever_2_turn_count_T": {
        "before": "sequential",
        "after": "by dependency rule",
        "turns_before": lever_2_sequential["turns_total"],
        "turns_after": lever_2_by_rule["turns_total"],
        "input_tokens_before": lever_2_sequential["tokens_in"],
        "input_tokens_after": lever_2_by_rule["tokens_in"],
        "cost_before_usd": lever_2_sequential["cost_usd"],
        "cost_after_usd": lever_2_by_rule["cost_usd"],
        "decisions_before": f"{lever_2_sequential['decisions_correct']}/42",
        "decisions_after": f"{lever_2_by_rule['decisions_correct']}/42",
    },
    "lever_3_observation_size_D": {
        "manual_tokens_before": 265,
        "manual_tokens_after": 540,
        "check_coverage_block_before": 32,
        "check_coverage_block_after": 307,
        "descriptor_delta_input_tokens": lever_3_descriptor_delta_tokens,
        "return_shape_delta_input_tokens": lever_3_return_shape_delta_tokens,
        "interpretation": "safety/interface improvement, not cost reduction",
    },
    "lever_4_success_rate": {
        "one_percentage_point_monthly_value_usd": one_pp_monthly,
        "headline_success_gap": headline_cheap["success_rate"] - headline_mid["success_rate"],
        "dominates": True,
        "reason": "At US$7.60 per failure and 8,000 claims/month, one pass-rate point is worth US$608/month, far more than the token bill.",
    },
}

print("Lever 1: fat tool block -> shipped input tokens", f"{lever1['fat_tool_block']['tokens_in']:,}", "->", f"{lever1['shipped']['tokens_in']:,}", "; removed", f"{lever1['delta_fat_minus_shipped']['tokens_in']:,}")
print("Lever 2: turns", lever_2_sequential["turns_total"], "->", lever_2_by_rule["turns_total"], "; input tokens", f"{lever_2_sequential['tokens_in']:,}", "->", f"{lever_2_by_rule['tokens_in']:,}")
print("Lever 3: manual 265 -> 540 tokens; descriptor alone +131,216 input tokens; return shape -1,751 input tokens")
print("Lever 4: one success-rate percentage point is", money(one_pp_monthly, 2), "per month")
print("Dominant lever: success rate")



## Section 7 - Caps and Optional Adjustments

State configured caps. Monthly per-user limit is not configured unless the team later defines a deployment assumption.

In [ ]:
CAPS = {
    "step_cap_turns": 12,
    "call_cap_model_calls": 22,
    "budget_ceiling_usd_per_run": 0.016,
    "monthly_limit_usd_per_user": MONTHLY_LIMIT_USD_PER_USER,
}

print("Caps to report:")
print(f"- Step cap: {CAPS['step_cap_turns']} turns")
print(f"- Call cap: {CAPS['call_cap_model_calls']} model calls")
print(f"- Budget ceiling: {money(CAPS['budget_ceiling_usd_per_run'])} per run")
print("- Monthly per-user limit:", "not set" if CAPS["monthly_limit_usd_per_user"] is None else money(CAPS["monthly_limit_usd_per_user"], 2))
print()
print("Caching/reasoning adjustments: none measured in the repository; baseline uses plain input/output tokens.")


## Section 8 - Write Result File and Report-Ready Summary

This section writes `results/d6_cost_model.json`. That file is the machine-readable D6 output for the report draft.

In [ ]:
report_ready_summary = (
    f"At Project A's 8,000 claims/month and US${FAILURE_USD:.2f} failure cost, "
    f"{headline_cheap['model']} is the lowest all-in v2 live option: "
    f"{pct(headline_cheap['success_rate'])} pass rate, {money(headline_cheap['cost_per_successful_claim_usd'])} per successful claim, "
    f"and {money(headline_cheap['monthly_total_usd'], 2)} per month. "
    f"Against {headline_mid['model']}, the cheap row clears break-even by {gap:.2%} "
    f"and changes monthly cost by {money(monthly_saving, 2)}. "
    f"The dominant lever is success rate: one percentage point is worth {money(one_pp_monthly, 2)} per month, "
    f"which overwhelms the token-level savings from tool and turn reductions."
)

result = {
    "assumptions": {
        "volume_claims_per_month": PROJECT_A_VOLUME,
        "failure_cost_usd": FAILURE_USD,
        "retrieval_usd_per_run": RETRIEVAL_USD,
        "tool_usd_per_run": TOOL_USD,
        "fixed_monthly_usd": FIXED_MONTHLY_USD,
        "fixed_monthly_note": FIXED_MONTHLY_NOTE,
        "monthly_limit_usd_per_user": MONTHLY_LIMIT_USD_PER_USER,
        "monthly_limit_note": MONTHLY_LIMIT_NOTE,
    },
    "model_rows": model_rows,
    "headline": headline,
    "sensitivity": sensitivity,
    "sensitivity_robust": robust,
    "cost_ledger": cost_ledger,
    "caps": CAPS,
    "report_ready_summary": report_ready_summary,
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {OUTPUT_JSON.relative_to(REPO)}")
print()
print("Report-ready summary:")
print(report_ready_summary)

## Section 9 - Checklist

Before copying results into the report, confirm:

- The headline comparison uses v2 live battery rows, not the v1 prompt comparison row.
- Layer 3 is either `0` / not measured or explicitly labelled as a deployment assumption.
- Monthly per-user limit is either `not set` or explicitly labelled as a deployment assumption.
- Lever 1 is labelled as a measured counterfactual prompt-prefix replay, not a live behaviour re-test.
- Caching/reasoning adjustments are not reported unless measured.